# Notebook 12 · Exploratory endpoints (added after the registered analysis)

**What it does:** computes further metrics, outside the registered plan, from the predictions that already exist.
It does not train or predict anything, and it does not touch the registered results (it writes a separate file):

- AP@0.75 (requires tighter boxes than AP@0.5);
- precision-recall curve and examination-level ROC curve;
- box precision, recall and F1, and examination-level sensitivity and specificity, at the operating point of one
  false positive per examination;
- box recall by size (small, medium, large);
- training loss per epoch of every model.

95% intervals by bootstrap over examinations (10,000 resamples, seed 20261029 for validation and 20261030 for the
test set), with the same draws for the three arms, as in the registered analysis.

**Before running:** the `src/ichdet` folder, version 0.5.0, on the Drive. No GPU is needed (*Runtime → Change
runtime type → CPU*). **Time:** about 30 to 60 minutes (almost all of it in the bootstrap).

In [ ]:
# 1. Drive and code
import json, os, sys, time, zipfile
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    RAIZ = Path('/content/drive/MyDrive/ich-benchmark')
except ImportError:                                  # local test: RAIZ mimics the Drive folder
    RAIZ = Path(os.environ['EXPERIMENTO_LOCAL'])

sys.path.insert(0, str(RAIZ / 'codigo'))
PREPARO = RAIZ / 'dados' / 'tc' / 'preparo'
PRIMEIRA = RAIZ / 'dados' / 'tc' / 'experimento' / 'completo'
CORRECAO = RAIZ / 'dados' / 'tc' / 'experimento' / 'correcao_retinanet'
SAIDA = CORRECAO / 'exploratorio'
SAIDA.mkdir(exist_ok=True)

import numpy as np
import ichdet
from ichdet import data_io, experiment, exploratory as ex
assert ichdet.__version__ >= '0.5.0', f'ichdet {ichdet.__version__}: copie de novo a pasta src/ichdet do repositório'
print('ichdet', ichdet.__version__)

In [ ]:
# 2. Reference boxes (read from inside the zip files) and stored predictions
A, B, C = 'retinanet', 'yolo_no_mosaic', 'yolo_mosaic'
BRACOS = (A, B, C)
PASTA = {A: CORRECAO, B: PRIMEIRA, C: PRIMEIRA}
K = experiment.FOLDS

def referencias(arquivo_zip, lista):
    with zipfile.ZipFile(PREPARO / arquivo_zip) as z:
        entradas = json.loads(z.read(lista))['validation']
    return {data_io.image_key(e['image']): (e['exam'], np.asarray(e['box'], dtype=float).reshape(-1, 4)) for e in entradas}

REF = {k: referencias('cq500_512.zip', f'monai/fold{k}.json') for k in K}
REF_TESTE = referencias('ctich_512.zip', 'monai/external.json')
PRED = {(k, b): data_io.load_predictions(experiment.job_dir(PASTA[b], k, b) / 'pred_validation.csv') for k in K for b in BRACOS}
PRED_TESTE = {(k, b): data_io.load_predictions(experiment.job_dir(PASTA[b], k, b) / 'pred_test.csv') for k in K for b in BRACOS}
print(f"validação: {sum(len(v) for v in REF.values())} cortes; teste: {len(REF_TESTE)} cortes")

In [ ]:
# 3. Matching tables and operating point of 1 false positive per exam (each model, on its own data)
T_VAL = {b: ex.build_tables([REF[k] for k in K], [PRED[(k, b)] for k in K]) for b in BRACOS}
T_TESTE = {b: ex.build_tables([REF_TESTE] * len(K), [PRED_TESTE[(k, b)] for k in K]) for b in BRACOS}
LIM_VAL = {b: ex.thresholds_1fp(T_VAL[b]) for b in BRACOS}
LIM_TESTE = {b: ex.thresholds_1fp(T_TESTE[b]) for b in BRACOS}

# check: AP@0.5 must equal the registered results
registrado = json.loads((CORRECAO / 'results_validation.json').read_text())['primary_A_vs_B']
for b, chave in ((A, 'A'), (B, 'B')):
    assert np.isclose(np.mean([t.ap50() for t in T_VAL[b]]), registrado[chave]['ap50']), b
print('AP@0.5 confere com os resultados registrados')

In [ ]:
# 4. Metrics with 95% intervals (bootstrap over exams; same draws for the three arms)
N, SEMENTE = 10_000, 20261029
inicio = time.time()
validacao = ex.summarise(T_VAL, LIM_VAL, lambda w: w, [len(t.exams) for t in T_VAL[A]], N, SEMENTE, reference_arm=B)
print(f'validação: {time.time() - inicio:.0f} s')
inicio = time.time()
n = len(T_TESTE[A][0].exams)
teste = ex.summarise(T_TESTE, LIM_TESTE, lambda w: [w[0]] * len(K), [n], N, SEMENTE + 1, reference_arm=B)
print(f'teste: {time.time() - inicio:.0f} s')

for nome, r in (('validação', validacao), ('teste', teste)):
    print(f'\n{nome}')
    for b in BRACOS:
        m = r['arms'][b]
        print(f"  {b:15s} " + '  '.join(f"{k} {m[k]['value']:.3f}" for k in ex.METRICS))

In [ ]:
# 5. Curves, box size and training loss
curvas = {}
for nome, tabelas in (('validation', T_VAL), ('test', T_TESTE)):
    curvas[nome] = {b: {'pr_iou50': [ex.pr_curve(t) for t in tabelas[b]],
                        'pr_iou75': [ex.pr_curve(t, 0.75) for t in tabelas[b]],
                        'roc_exam': [ex.roc_curve(t) for t in tabelas[b]]} for b in BRACOS}
tamanho = {'validation': {b: ex.size_sensitivity([(REF[k], PRED[(k, b)]) for k in K], LIM_VAL[b]) for b in BRACOS},
           'test': {b: ex.size_sensitivity([(REF_TESTE, PRED_TESTE[(k, b)]) for k in K], LIM_TESTE[b]) for b in BRACOS}}
loss = {b: {f'fold{k}': ex.loss_history(experiment.job_dir(PASTA[b], k, b), b) for k in K} for b in BRACOS}
for b in BRACOS:
    print(b, {s: f"{v['detected']}/{v['boxes']}" for s, v in tamanho['validation'][b].items()},
          'loss fold 1:', round(loss[b]['fold1']['loss'][0], 3), '→', round(loss[b]['fold1']['loss'][-1], 3))

resultado = {
    'note': 'Exploratory endpoints, not in the registered plan (supervision meeting of 5 Nov 2026).',
    'computed': time.strftime('%Y-%m-%d %H:%M:%S'), 'ichdet_version': ichdet.__version__,
    'operating_point': '1 false positive per examination, per fold model on its own data',
    'size_bins_px2': {s: [lo, hi if np.isfinite(hi) else None] for s, lo, hi in ex.SIZE_BINS},
    'recall_points': list(np.linspace(0, 1, 101)), 'roc_fpr_grid': list(ex.ROC_FPR_GRID),
    'validation': validacao, 'test': teste, 'curves': curvas, 'size_sensitivity': tamanho, 'training_loss': loss,
}
(SAIDA / 'exploratory_results.json').write_text(json.dumps(resultado, indent=1, default=float))
print('gravado em', SAIDA / 'exploratory_results.json')

## After running

1. Save the executed notebook (*File → Download → .ipynb*).
2. Download `exploratory_results.json` from `dados/tc/experimento/correcao_retinanet/exploratorio/` on the Drive;
   it is the file `data/correcao_retinanet/exploratory_results.json` of this repository, read by `tools/make_figures.py`.